# CSV + API

In this reboot, we are going to use:

- The [Goodreads books](https://www.kaggle.com/jealousleopard/goodreadsbooks) dataset from Kaggle.
- The [Open Library Books API](https://openlibrary.org/dev/docs/api/books)

The goal of this livecode is to load the data from a CSV + loop over rows to enrich each row with information such as:

- List of subjects (Science, Humor, Travel, etc.)
- The cover URL of the book
- Other information you'd find useful in the JSON API

First, download the CSV in the local folder:

In [1]:
!curl -L https://wagon-public-datasets.s3.amazonaws.com/02-Data-Toolkit/02-Data-Sourcing/books.csv > books.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1509k  100 1509k    0     0   498k      0  0:00:03  0:00:03 --:--:--  498k


In [2]:
!ls -lh

total 4248
-rw-r--r--@ 1 scooter  staff   1.5M Oct  8 12:48 books.csv
-rw-r--r--@ 1 scooter  staff   579B Jan 18  2026 README.md
-rw-r--r--@ 1 scooter  staff   4.0K Oct  8 12:46 Recap.ipynb


Then import the usual suspects!

In [3]:
import requests
import pandas as pd
import numpy as np

## Load books from CSV

In [9]:
books_df = pd.read_csv('books.csv')
books_df.head(3)

books_df.loc[0, 'isbn']

'0439785960'

Let's add a new column

# Add `cover_url` from the API

In [22]:
bibkeys = ','.join([f"ISBN:{isbn}" for isbn in list(books_df['isbn'])])

url = f"http://openlibrary.org/api/books.json?bibkeys={bibkeys}"
# requests.get(url).json()
url

'http://openlibrary.org/api/books.json?bibkeys=ISBN:0439785960,ISBN:0439358078,ISBN:0439554934,ISBN:0439554896,ISBN:043965548X,ISBN:0439682584,ISBN:0976540606,ISBN:0439827604,ISBN:0517226952,ISBN:0345453743,ISBN:1400052920,ISBN:0739322206,ISBN:0517149257,ISBN:076790818X,ISBN:0767915062,ISBN:0767910435,ISBN:0767903862,ISBN:076790382X,ISBN:0060920084,ISBN:0380713802,ISBN:0380727501,ISBN:0380715430,ISBN:0345538374,ISBN:0618517650,ISBN:0618346244,ISBN:0618346252,ISBN:0618260587,ISBN:0618391002,ISBN:0618510826,ISBN:0618153977,ISBN:193337201X,ISBN:097669400X,ISBN:0689840926,ISBN:1557344493,ISBN:0385326505,ISBN:1575606240,ISBN:1595580271,ISBN:1595962808,ISBN:0670059676,ISBN:0141312629,ISBN:0595321801,ISBN:1590301943,ISBN:0449146979,ISBN:0061159174,ISBN:006076273X,ISBN:0060749911,ISBN:0273704745,ISBN:1932386106,ISBN:096513671X,ISBN:0374517193,ISBN:0374280398,ISBN:0374519749,ISBN:0374522596,ISBN:0374518734,ISBN:0374522871,ISBN:0374519323,ISBN:0374516006,ISBN:0374520658,ISBN:0822205106,ISBN:0679

## API - Open Library

In [43]:
from IPython.display import Image, display

# Figure out how we can get the cover url (search by name? isb? author?)
def get_cover_url(isbn):
    url = f"http://openlibrary.org/api/books.json?bibkeys=ISBN:{isbn}&jscmd=data&format=json"
    data = requests.get(url).json()
    if data[f"ISBN:{isbn}"] == None or data[f"ISBN:{isbn}"]['cover'] == None:
        return None

    return data[f"ISBN:{isbn}"]['cover']['medium']

Image(url=get_cover_url(9780980200447))

# Make a method that would programmatically get the url from the API (batch request would be nice)

In [47]:
for index, row in books_df.head(10).iterrows():
    isbn = row["isbn"]
    url = get_cover_url(isbn)
    if url == None:
        print('No image attached')

    books_df.loc[index, 'cover_url'] = url
    # display(Image(url=url))

In [48]:
books_df

,bookID,title,authors,average_rating,isbn,isbn13,language_code,# num_pages,ratings_count,text_reviews_count,cover_url
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling-Mary GrandPré,4.56,0439785960,9780439785969,eng,652,1944099,26249,https://covers.openlibrary.org/b/id/15156081-M...
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling-Mary GrandPré,4.49,0439358078,9780439358071,eng,870,1996446,27613,https://covers.openlibrary.org/b/id/12025650-M...
2,3,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling-Mary GrandPré,4.47,0439554934,9780439554930,eng,320,5629932,70390,https://covers.openlibrary.org/b/id/7572543-M.jpg
3,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.41,0439554896,9780439554893,eng,352,6267,272,https://covers.openlibrary.org/b/id/10301720-M...
4,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling-Mary GrandPré,4.55,043965548X,9780439655484,eng,435,2149872,33964,https://covers.openlibrary.org/b/id/8778528-M.jpg
...,...,...,...,...,...,...,...,...,...,...,...
13714,47699,M Is for Magic,Neil Gaiman-Teddy Kristiansen,3.82,0061186422,9780061186424,eng,260,11317,1060,NaN
13715,47700,Black Orchid,Neil Gaiman-Dave McKean,3.72,0930289552,9780930289553,eng,160,8710,361,NaN
13716,47701,InterWorld (InterWorld #1),Neil Gaiman-Michael Reaves,3.53,0061238961,9780061238963,en-US,239,14334,1485,NaN
13717,47708,The Faeries' Oracle,Brian Froud-Jessica Macbeth,4.43,0743201116,9780743201117,eng,224,1550,38,NaN


## Calling the API with multiple ISBNs at a time

In [79]:
# Take a list of isbn, then return a dict of cover url
# { '123123': 'https://covers.openlibrary.org/b/id/7572543-S.jpg' }


def get_cover_urls(isbns):
    bibkeys = ','.join([f"ISBN:{isbn}" for isbn in isbns])

    url = f"http://openlibrary.org/api/books.json?bibkeys={bibkeys}&jscmd=data&format=json"
    # requests.get(url).json()
    data = requests.get(url).json()
    return data
    # if data[f"ISBN:{isbn}"] == None or data[f"ISBN:{isbn}"]["cover"] == None:
    #     return None

    # return data[f"ISBN:{isbn}"]["cover"]["medium"]


isbns = books_df['isbn'][:40]
books = get_cover_urls(isbns)

In [81]:
books

{'ISBN:0439785960': {'url': 'http://openlibrary.org/books/OL24280830M/Harry_Potter_and_the_Half-Blood_Prince',
  'key': '/books/OL24280830M',
  'title': 'Harry Potter and the Half-Blood Prince',
  'authors': [{'url': 'http://openlibrary.org/authors/OL23919A/J._K._Rowling',
    'name': 'J. K. Rowling'}],
  'number_of_pages': 652,
  'pagination': '652p',
  'identifiers': {'amazon': ['0439785960'],
   'goodreads': ['53178655'],
   'isbn_10': ['0439785960'],
   'isbn_13': ['9780439785969'],
   'oclc': ['70666878', '819153929'],
   'openlibrary': ['OL24280830M']},
  'classifications': {'lc_classifications': ['PZ7.R79835Halc 2005']},
  'publishers': [{'name': 'Scholastic'}],
  'publish_places': [{'name': 'New York, USA'}],
  'publish_date': '2006-09',
  'subjects': [{'name': 'orphans',
    'url': 'https://openlibrary.org/subjects/orphans'},
   {'name': 'foster homes',
    'url': 'https://openlibrary.org/subjects/foster_homes'},
   {'name': 'romans', 'url': 'https://openlibrary.org/subjects/r

In [95]:
cover_urls = {}

for isbn in books.keys():
    actual_isbn = isbn.split(':')[1]

    if 'cover' in books[isbn].keys():
        cover_urls[actual_isbn] = books[isbn]['cover']['medium']
    else:
        cover_urls[actual_isbn] = None

cover_urls

{'0439785960': 'https://covers.openlibrary.org/b/id/15156081-M.jpg',
 '0439358078': 'https://covers.openlibrary.org/b/id/12025650-M.jpg',
 '0439554934': 'https://covers.openlibrary.org/b/id/7572543-M.jpg',
 '0439554896': 'https://covers.openlibrary.org/b/id/10301720-M.jpg',
 '043965548X': 'https://covers.openlibrary.org/b/id/8778528-M.jpg',
 '0439682584': 'https://covers.openlibrary.org/b/id/278981-M.jpg',
 '0976540606': 'https://covers.openlibrary.org/b/id/742235-M.jpg',
 '0439827604': 'https://covers.openlibrary.org/b/id/279436-M.jpg',
 '0517226952': 'https://covers.openlibrary.org/b/id/12617870-M.jpg',
 '0345453743': 'https://covers.openlibrary.org/b/id/14656530-M.jpg',
 '1400052920': 'https://covers.openlibrary.org/b/id/10176291-M.jpg',
 '0739322206': 'https://covers.openlibrary.org/b/id/8769632-M.jpg',
 '0517149257': 'https://covers.openlibrary.org/b/id/7892560-M.jpg',
 '076790818X': 'https://covers.openlibrary.org/b/id/12995805-M.jpg',
 '0767915062': 'https://covers.openlibrary.o

In [102]:
!uv add tqdm

Resolved 50 packages in 200ms                                        
Installed 1 package in 6ms                                       ░░░░░░░░░░░░░░░░░░░░ [0/0] Installing wheels...                                 
 + tqdm==4.70.1


In [ ]:
from tqdm import tqdm

cover_urls = {}

for isbns in tqdm(np.array_split(books_df['isbn'], 300)):
  books = get_cover_urls(isbns)
  for isbn in books.keys():
      actual_isbn = isbn.split(":")[1]

      if "cover" in books[isbn].keys():
          cover_urls[actual_isbn] = books[isbn]["cover"]["medium"]
      else:
          cover_urls[actual_isbn] = None

cover_urls

KeyboardInterrupt: 